# gpu-free-serve — Kaggle

Boots an LLM on this notebook's free GPU and exposes an OpenAI-compatible
endpoint you can call from your own machine.

**Before running:** Settings > Accelerator > GPU T4 x2 (and Internet: On)

Then run the cells top to bottom. The last cell keeps running — that is the
server. Closing this tab kills the endpoint.

In [ ]:
!nvidia-smi

In [ ]:
# Clone the repo (or update it if it is already here).
import os, subprocess

os.chdir("/kaggle/working")
if os.path.isdir("gpu-free-serve"):
    subprocess.run(["git", "-C", "gpu-free-serve", "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/iramarfalcao/gpu-free-serve.git"], check=True)
os.chdir("/kaggle/working/gpu-free-serve")
print(os.getcwd())

In [ ]:
# Secrets are read from the platform's secret manager and never written into
# this notebook. Kaggle: Add-ons > Secrets (and turn Internet ON in the right sidebar).
#
#   GITHUB_TOKEN     scope `gist` -> lets `gpufree connect --gist` work
#   HF_TOKEN         only for gated models (Llama, Gemma, ...)
#   NGROK_AUTHTOKEN  only if TUNNEL = "ngrok"
#   GPUFREE_API_KEY  optional fixed API key; otherwise a random one is generated
import os
from kaggle_secrets import UserSecretsClient

_client = UserSecretsClient()

def read_secret(name):
    try:
        return _client.get_secret(name)
    except Exception:
        return None

for name in ("GITHUB_TOKEN", "HF_TOKEN", "NGROK_AUTHTOKEN", "GPUFREE_API_KEY"):
    value = read_secret(name)
    if value:
        os.environ[name] = value
        print(f"{name}: loaded")
    else:
        print(f"{name}: not set (fine if you don't need it)")

## Pick a model

The catalog below comes from `models.yaml` in the repo. Edit that file (and push)
to add your own models — or set `MODEL` to a raw id:

* `Qwen/Qwen2.5-7B-Instruct-AWQ` → served by vLLM
* `ollama:llama3.1:8b` → served by Ollama

In [ ]:
!python -m gpufree.server.serve --list

In [ ]:
MODEL = "qwen2.5-7b-awq"   # a catalog name, `org/model`, or `ollama:tag`
TUNNEL = "cloudflare"      # "cloudflare" (no account) or "ngrok" (needs authtoken)
PUBLISH = "gist"           # "gist" publishes the endpoint; "none" just prints it

# This cell blocks while the server is alive. First run installs vLLM and
# downloads the weights (10-15 min).
!python -m gpufree.server.serve --model {MODEL} --tunnel {TUNNEL} --publish {PUBLISH}

## Use it from your machine

```bash
pipx install git+https://github.com/iramarfalcao/gpu-free-serve.git   # once

gpufree connect --gist          # or: gpufree connect --url <url> --key <key>
gpufree status
gpufree chat "explain attention in two sentences"
gpufree proxy                   # http://127.0.0.1:8787/v1 — no key needed
```

Point any OpenAI client at the proxy:

```python
from openai import OpenAI
client = OpenAI(base_url="http://127.0.0.1:8787/v1", api_key="not-needed")
```

**Do not commit this notebook with its output** — the banner prints the API key.
Edit > Clear all outputs first.